In [ ]:
import pandas as pd
import numpy as np

print("Boîte à outils prête !")

In [ ]:
# 1. Lecture du fichier DVF+ (Données récentes)
# Note : on utilise sep='|' car c'est le délimiteur que tu utilisais en R
df_recent = pd.read_csv("/content/dvf_plus.csv", sep="|", low_memory=False)

# 2. Lecture du fichier du professeur (Données 2010-2013 au format Parquet)
# Le format Parquet est très rapide et léger
df_old = pd.read_parquet("/content/dv3f_filtered_2010_2013.parquet")

print(f"Données chargées ! Récent : {df_recent.shape} lignes, Ancien : {df_old.shape} lignes.")

In [ ]:
# On recharge df_recent pour repartir sur une base propre
df_recent = pd.read_csv("/content/dvf_plus.csv", sep="|", low_memory=False)

# Espace de travail : Côte d'Azur (Départements 06 et 83)
# On convertit en string pour être sûr de la comparaison
df_recent['coddep'] = df_recent['coddep'].astype(str).str.zfill(2)
depts_cote_azur = ['06', '83']

df_recent = df_recent[df_recent['coddep'].isin(depts_cote_azur)].copy()

# ETAPE CRUCIALE : Calcul du prix au m2
def clean_prices(df, col_prix, col_surface):
    # Filtrage des surfaces nulles et calcul
    df = df[df[col_surface] > 0].copy()
    df['price_sqm'] = df[col_prix] / df[col_surface]
    # Filtre sur le prix au m2 (entre 1000€ et 40000€)
    df = df[(df['price_sqm'] > 1000) & (df['price_sqm'] < 40000)]
    df['y'] = np.log(df['price_sqm'])
    return df

# Application du nettoyage avec les noms de colonnes vérifiés
df_recent = clean_prices(df_recent, 'valeurfonc', 'sbati')

print(f"Filtrage terminé. Nouveau volume pour 06/83 : {df_recent.shape[0]} lignes.")
if df_recent.shape[0] > 0:
    display(df_recent[['coddep', 'valeurfonc', 'sbati', 'price_sqm', 'y']].head())

In [ ]:
print("Colonnes Récent :", df_recent.columns.tolist())
print("Colonnes Prof :", df_old.columns.tolist())

In [ ]:
# Inspection pour comprendre pourquoi on a 0 lignes
print("Valeurs uniques dans 'coddep' :", df_recent['coddep'].unique()[:5] if 'coddep' in df_recent.columns else 'Colonne absente')

# On recharge proprement pour voir les colonnes de prix/surface sans filtre
df_check = pd.read_csv('/content/dvf_plus.csv', sep='|', low_memory=False, nrows=100)
print("Colonnes de prix/surface disponibles :", [c for c in df_check.columns if 'val' in c or 'surf' in c or 'bati' in c])
print("Aperçu des 5 premières lignes de prix/surface :")
display(df_check.filter(like='val').head())
display(df_check.filter(like='bati').head())

In [ ]:
# 1. Nettoyage du fichier Prof (df_old)
df_old['coddep'] = df_old['coddep'].astype(str).str.zfill(2)
df_old_azur = df_old[df_old['coddep'].isin(['06', '83'])].copy()
df_old_azur = clean_prices(df_old_azur, 'valeurfonc', 'sbati')

# 2. On ajoute 'l_codinsee' aux colonnes communes pour l'effet fixe commune
cols_communes = ['anneemut', 'moismut', 'coddep', 'valeurfonc', 'sbati', 'price_sqm', 'y', 'libtypbien', 'l_codinsee']

# Fusion
df_total = pd.concat([df_recent[cols_communes], df_old_azur[cols_communes]], axis=0)

# Création de la variable temps
df_total['t'] = df_total['anneemut'].astype(str) + "-" + df_total['moismut'].astype(str).str.zfill(2)

print(f"Fusion terminée avec communes ! Total : {df_total.shape[0]} lignes.")

In [ ]:
import statsmodels.formula.api as smf

# --- VÉRIFICATION DES LABELS ---
# On regarde comment sont écrits les types de biens pour éviter les erreurs de casse
print("Types de biens disponibles :", df_total['libtypbien'].unique())

# --- SÉPARATION DES MARCHÉS ---
# On utilise une recherche insensible à la casse pour être sûr de trouver les appartements
df_appat = df_total[df_total['libtypbien'].str.contains('appart', case=False, na=False)].copy()

if df_appat.empty:
    print("ERREUR : Aucun appartement trouvé. Le modèle ne peut pas être estimé.")
else:
    # --- PRÉPARATION DES EFFETS FIXES ---
    df_appat['dept_year'] = df_appat['coddep'].astype(str) + "_" + df_appat['anneemut'].astype(str)

    # On utilise 'l_codinsee' comme identifiant de commune
    threshold = 50
    counts = df_appat['l_codinsee'].value_counts()
    df_appat['commune_ref'] = df_appat['l_codinsee'].apply(lambda x: x if counts[x] >= threshold else "TINY")

    # --- ESTIMATION DU MODÈLE ---
    # On vérifie qu'on a bien plusieurs périodes et communes pour que le modèle soit identifiable
    if df_appat['t'].nunique() > 1 and df_appat['commune_ref'].nunique() > 1:
        formula = "y ~ sbati + C(t) + C(commune_ref) + C(dept_year)"
        model_appat = smf.ols(formula, data=df_appat).fit()

        # --- EXTRACTION DU PRIX NET ---
        beta_surface = model_appat.params['sbati']
        df_appat['log_p_net'] = df_appat['y'] - (beta_surface * df_appat['sbati'])

        print(f"Modèle estimé avec succès sur {len(df_appat)} lignes.")
        print(f"R-carré : {model_appat.rsquared:.4f}")
    else:
        print("Pas assez de variations (temps/communes) pour estimer le modèle.")

In [ ]:
# 1. Harmonisation du fichier Prof (df_old)
# On s'assure que les départements sont au bon format et on filtre la Côte d'Azur
df_old['coddep'] = df_old['coddep'].astype(str).str.zfill(2)
df_old_azur = df_old[df_old['coddep'].isin(['06', '83'])].copy()

# Calcul de y pour le fichier prof (comme tu l'as fait pour le récent)
df_old_azur['price_sqm'] = df_old_azur['valeurfonc'] / df_old_azur['sbati']
# Nettoyage des prix aberrants (Outliers) pour la rigueur statistique
df_old_azur = df_old_azur[(df_old_azur['price_sqm'] > 1000) & (df_old_azur['price_sqm'] < 40000)]
df_old_azur['y'] = np.log(df_old_azur['price_sqm'])

# 2. Unification (On ne garde que les colonnes essentielles pour le modèle)
# Je renomme tes colonnes pour qu'elles correspondent entre les deux fichiers
cols_to_keep = ['anneemut', 'moismut', 'coddep', 'y', 'sbati', 'libtypbien', 'l_codinsee']

# Création du dataframe final unifié
df_final = pd.concat([
    df_recent[['anneemut', 'moismut', 'coddep', 'y', 'sbati', 'libtypbien', 'l_codinsee']],
    df_old_azur[['anneemut', 'moismut', 'coddep', 'y', 'sbati', 'libtypbien', 'l_codinsee']]
], axis=0)

# Création de la variable temporelle t (Mois-Année) demandée par le prof
df_final['t'] = df_final['anneemut'].astype(str) + "-" + df_final['moismut'].astype(str).str.zfill(2)

print(f"Base de données unifiée : {len(df_final)} transactions prêtes pour l'analyse.")

In [ ]:
import statsmodels.formula.api as smf
import pandas as pd
import numpy as np
import os

# --- RECONSTRUCTION COMPLÈTE CORRIGÉE ---
def reload_and_prepare():
    print("🔄 Rechargement et préparation des données...")
    # Chargement
    df_r = pd.read_csv('/content/dvf_plus.csv', sep='|', low_memory=False)
    df_o = pd.read_parquet('/content/dv3f_filtered_2010_2013.parquet')

    # Nettoyage et filtrage Côte d'Azur
    depts = ['06', '83']
    df_r['coddep'] = df_r['coddep'].astype(str).str.zfill(2)
    df_r = df_r[df_r['coddep'].isin(depts)].copy()

    df_o['coddep'] = df_o['coddep'].astype(str).str.zfill(2)
    df_o = df_o[df_o['coddep'].isin(depts)].copy()

    # Calcul des prix et log (Correction de l'assignation)
    def prepare_df(df):
        df = df[df['sbati'] > 0].copy()
        df['price_sqm'] = df['valeurfonc'] / df['sbati']
        df = df[(df['price_sqm'] > 1000) & (df['price_sqm'] < 40000)]
        df['y'] = np.log(df['price_sqm'])
        return df

    df_r = prepare_df(df_r)
    df_o = prepare_df(df_o)

    # Fusion
    cols = ['anneemut', 'moismut', 'coddep', 'y', 'sbati', 'libtypbien', 'l_codinsee']
    df_f = pd.concat([df_r[cols], df_o[cols]], axis=0)
    df_f['t'] = df_f['anneemut'].astype(str) + "-" + df_f['moismut'].astype(str).str.zfill(2)
    return df_f

if 'df_final' not in globals():
    df_final = reload_and_prepare()

# 1. Filtrer pour les APPARTEMENTS uniquement
df_appat = df_final[df_final['libtypbien'].str.contains('appart', case=False, na=False)].copy()

if df_appat.empty:
    print("ERREUR : Aucun appartement trouvé.")
else:
    # 2. Gestion des communes et effets fixes
    seuil = 30
    counts = df_appat['l_codinsee'].value_counts()
    df_appat['commune_ref'] = df_appat['l_codinsee'].where(df_appat['l_codinsee'].map(counts) >= seuil, 'TINY')
    df_appat['dept_year'] = df_appat['coddep'].astype(str) + "_" + df_appat['anneemut'].astype(str)

    # 3. Estimation
    print("⏳ Estimation du modèle hédonique...")
    formula = "y ~ sbati + C(t) + C(commune_ref) + C(dept_year)"
    model_appat = smf.ols(formula, data=df_appat).fit()

    # 4. Calcul du prix net
    beta_sbati = model_appat.params['sbati']
    df_appat['log_p_net'] = df_appat['y'] - (beta_sbati * df_appat['sbati'])

    print(f"✅ Modèle estimé sur {len(df_appat)} transactions. R²: {model_appat.rsquared:.4f}")

In [ ]:
import statsmodels.formula.api as smf
import pandas as pd
import numpy as np

# --- VÉRIFICATION DES DONNÉES ---
if 'df_final' not in globals():
    print("🔄 Reconstruction de df_final...")
    # On s'assure que les briques de base sont là
    if 'df_recent' in globals() and 'df_old_azur' in globals():
        cols_to_keep = ['anneemut', 'moismut', 'coddep', 'y', 'sbati', 'libtypbien', 'l_codinsee']
        df_final = pd.concat([
            df_recent[cols_to_keep],
            df_old_azur[cols_to_keep]
        ], axis=0)
        df_final['t'] = df_final['anneemut'].astype(str) + "-" + df_final['moismut'].astype(str).str.zfill(2)

# 1. Filtrer pour les APPARTEMENTS uniquement
if 'df_final' in globals():
    df_appat = df_final[df_final['libtypbien'].str.contains('appart', case=False, na=False)].copy()

    if df_appat.empty:
        print("❌ ERREUR : Aucun appartement trouvé.")
    else:
        # 2. Gestion des petites communes
        seuil = 30
        counts = df_appat['l_codinsee'].value_counts()
        df_appat['commune_ref'] = df_appat['l_codinsee'].where(df_appat['l_codinsee'].map(counts) >= seuil, 'TINY')

        # 3. Création de l'interaction département-année
        df_appat['dept_year'] = df_appat['coddep'].astype(str) + "_" + df_appat['anneemut'].astype(str)

        # 4. Estimation du modèle OLS
        print("⏳ Estimation du modèle en cours (cela peut prendre 10-20 secondes)...")
        formula = "y ~ sbati + C(t) + C(commune_ref) + C(dept_year)"
        model_appat = smf.ols(formula, data=df_appat).fit()

        # 5. Calcul du prix net
        beta_sbati = model_appat.params['sbati']
        df_appat['log_p_net'] = df_appat['y'] - (beta_sbati * df_appat['sbati'])

        print(f"✅ Modèle estimé sur {len(df_appat)} transactions.")
        print(f"R-carré : {model_appat.rsquared:.4f}")
        display(df_appat[['l_codinsee', 't', 'log_p_net']].head())
else:
    print("❌ ERREUR : df_final est introuvable. Veuillez exécuter la cellule z_ZNd4SdshNu d'abord.")

In [ ]:
# --- 4.1. Calcul des résidus pour les communes "TINY" ---
# e_i = y_réel - log_p_net [cite: 172]
df_appat['residu'] = df_appat['y'] - df_appat['log_p_net']

# Calcul de la correction moyenne par commune TINY [cite: 172]
tiny_correction = df_appat[df_appat['commune_ref'] == 'TINY'].groupby('l_codinsee')['residu'].mean()

# --- 4.2. Agrégation par Commune et par Mois ---
# On agrège en prenant la médiane du log_p_net [cite: 161]
indices_bruts = df_appat.groupby(['l_codinsee', 't'])['log_p_net'].median().reset_index()

# --- 4.3. Application de la correction ---
def apply_tiny_correction(row):
    if row['l_codinsee'] in tiny_correction:
        return row['log_p_net'] + tiny_correction[row['l_codinsee']] # [cite: 174]
    return row['log_p_net']

indices_bruts['log_p_net_corr'] = indices_bruts.apply(apply_tiny_correction, axis=1)

# --- 4.4. Passage en Indice Base 100 ---
# On choisit une période de référence (t0), par exemple Janvier 2010 [cite: 199]
t0 = "2010-01"

def compute_index(group):
    # Trouver le prix de référence au temps t0 pour cette commune
    ref_val = group.loc[group['t'] == t0, 'log_p_net_corr']
    if not ref_val.empty:
        log_p_t0 = ref_val.values[0]
        # Formule de l'indice : 100 * exp(log P_t - log P_t0) [cite: 201]
        group['Index'] = 100 * np.exp(group['log_p_net_corr'] - log_p_t0)
    else:
        group['Index'] = np.nan # Pour les périodes sans transactions
    return group

df_indices_finaux = indices_bruts.groupby('l_codinsee').apply(compute_index)

print("L'indice de prix par commune a été calculé avec succès !")

In [ ]:
import pandas as pd
from itertools import product

# --- VÉRIFICATION DE SÉCURITÉ ---
if 'df_appat' not in globals() or 'df_indices_finaux' not in globals():
    print("❌ ERREUR : Les données des appartements ou des indices sont manquantes.")
    print("👉 SOLUTION : Exécutez d'abord les cellules '9lqXI8xFsxB1' et '1iLix2SmttGu' au-dessus.")
else:
    # 1. Créer la liste de tous les mois et communes possibles
    all_months = sorted(df_appat['t'].unique())
    all_communes = df_appat['l_codinsee'].unique()

    # 2. Créer une grille complète (chaque commune x chaque mois)
    grille = pd.DataFrame(list(product(all_communes, all_months)), columns=['l_codinsee', 't'])

    # 3. Fusionner avec les indices calculés
    # On reset l'index pour éviter les conflits de colonnes lors du merge
    df_indices_clean = df_indices_finaux.reset_index(drop=True)
    df_complet = pd.merge(grille, df_indices_clean, on=['l_codinsee', 't'], how='left')

    # 4. Remplissage par interpolation linéaire pour combler les mois sans ventes
    # 'both' permet de remplir aussi le début et la fin de la série
    df_complet['Index_Final'] = df_complet.groupby('l_codinsee')['Index'].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))

    print("✅ La série temporelle est maintenant complète et continue !")
    display(df_complet.head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(14, 7))

# On choisit Nice (06088) et Antibes (06004) pour l'exemple
communes_stars = ['06088', '06004']

for code in communes_stars:
    data_plot = df_complet[df_complet['l_codinsee'] == code]
    plt.plot(data_plot['t'], data_plot['Index_Final'], label=f"Commune {code}")

plt.title("Évolution des prix immobiliers (Base 100 en Janvier 2010)", fontsize=15)
plt.xlabel("Temps (Mois-Année)")
plt.ylabel("Indice de prix")
plt.xticks(df_complet['t'].unique()[::12], rotation=45) # On affiche un label par an
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 1. Calcul de la moyenne du d)partement (Benchmark)
# On regroupe par mois 't' et on calcule la moyenne des indices de toutes les communes
# Note : Assurez-vous d'avoir ex)cut) la cellule Wc0J41D8uB_z avant celle-ci.
if 'df_complet' in globals():
    benchmark_dept = df_complet.groupby('t')['Index_Final'].mean().reset_index()
    print("Benchmark d)partemental calcul) !")
else:
    print("❌ ERREUR : df_complet est absent. Remontez   la cellule Wc0J41D8uB_z et ex)cutez-la.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration du style
plt.figure(figsize=(15, 8))
sns.set_style("whitegrid")

# 2. Liste des communes à afficher (Codes INSEE)
# 06088 = Nice, 06004 = Antibes, 06029 = Cannes
communes_a_tracer = ['06088', '06004', '06029']
noms_communes = {'06088': 'Nice', '06004': 'Antibes', '06029': 'Cannes'}

# 3. Tracer les courbes des communes
for code in communes_a_tracer:
    df_temp = df_complet[df_complet['l_codinsee'] == code]
    plt.plot(df_temp['t'], df_temp['Index_Final'], label=noms_communes[code], linewidth=2)

# 4. Tracer le Benchmark Départemental (Ligne de référence)
plt.plot(benchmark_dept['t'], benchmark_dept['Index_Final'],
         label="Moyenne Départementale (06/83)",
         color='black', linestyle='--', linewidth=3)

# 5. Mise en forme "20/20"
plt.title("Évolution des prix immobiliers à qualité constante (Base 100 en Janvier 2010)", fontsize=16, fontweight='bold')
plt.xlabel("Période (Mois-Année)", fontsize=12)
plt.ylabel("Indice de prix ($Index_{c,t}$)", fontsize=12)

# Afficher une date sur 12 pour ne pas surcharger l'axe X
plt.xticks(df_complet['t'].unique()[::12], rotation=45)

plt.axhline(y=100, color='gray', linestyle=':', alpha=0.7) # Ligne de base 100
plt.legend(fontsize=11)
plt.tight_layout()

# Sauvegarde pour le rapport
plt.savefig("courbe_temporelle_immobilier.png", dpi=300)
plt.show()

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

# 1. Téléchargement des frontières des communes françaises (simplifiées pour la rapidité)
url = "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/communes-avec-outre-mer.geojson"
france_map = gpd.read_file(url)

# 2. Filtrer pour la Côte d'Azur (06 et 83)
azur_map = france_map[france_map['code'].str.startswith(('06', '83'))].copy()

# 3. Choisir une date pour la carte (ex: la dernière date disponible dans tes données)
date_carte = df_complet['t'].max()
data_last_date = df_complet[df_complet['t'] == date_carte]

# 4. Fusionner les données de l'indice avec la carte
# On joint sur le code INSEE (l_codinsee dans tes données, 'code' dans la carte)
map_final = azur_map.merge(data_last_date, left_on='code', right_on='l_codinsee')

# 5. Création de la carte "Major de Promo"
fig, ax = plt.subplots(1, 1, figsize=(15, 10))
map_final.plot(column='Index_Final',
               ax=ax,
               legend=True,
               cmap='OrRd', # Dégradé Orange-Rouge
               legend_kwds={'label': f"Niveau de l'indice en {date_carte} (Base 100 en 2010)"},
               missing_kwds={"color": "lightgrey"}) # Communes sans données en gris

ax.set_title(f"Répartition spatiale des prix immobiliers sur la Côte d'Azur\n(Indice à qualité constante - {date_carte})", fontsize=15)
ax.set_axis_off() # On retire les axes de coordonnées pour faire propre
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Sélectionner les 30 communes avec le plus de transactions pour la lisibilité
top_communes = df_appat['l_codinsee'].value_counts().nlargest(30).index
df_heatmap = df_complet[df_complet['l_codinsee'].isin(top_communes)]

# 2. Pivoter les données : Communes en lignes, Temps en colonnes
# On utilise l'Index_Final (base 100 en 2010)
pivot_table = df_heatmap.pivot(index='l_codinsee', columns='t', values='Index_Final')

# 3. Création de la Heatmap
plt.figure(figsize=(20, 10))
sns.heatmap(pivot_table,
            cmap='YlOrRd', # Jaune -> Orange -> Rouge
            cbar_kws={'label': 'Indice (Base 100 en 2010)'}, # Correction de cbar_kds en cbar_kws
            xticklabels=12) # On affiche un label de temps tous les 12 mois

plt.title("Heatmap des dynamiques immobilières : Indices par commune et par mois", fontsize=18, fontweight='bold')
plt.xlabel("Temps (Mois-Année)", fontsize=14)
plt.ylabel("Code INSEE de la commune", fontsize=14)

# Optionnel : Remplacer les codes INSEE par les noms des communes si tu as un dictionnaire
# plt.yticks(range(len(top_communes)), [noms_communes.get(c, c) for c in pivot_table.index])

plt.tight_layout()
plt.savefig("heatmap_immobilier_azur.png", dpi=300)
plt.show()

In [ ]:
import geopandas as gpd

# 1. Charger la carte si elle n'est pas en mémoire
url = "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/communes-avec-outre-mer.geojson"
france_map = gpd.read_file(url)

# 2. Simplifier les contours de la carte
# Le chiffre 0.01 détermine le niveau de simplification
france_map['geometry'] = france_map['geometry'].simplify(0.01)

print("Contours simplifiés avec succès !")

In [ ]:
import plotly.express as px
import pandas as pd

# --- VÉRIFICATION DE SÉCURITÉ ---
if 'df_complet' not in globals():
    print("❌ ERREUR : La variable 'df_complet' est introuvable.")
    print("👉 SOLUTION : Re-exécute la cellule Wc0J41D8uB_z juste au-dessus pour recréer les données.")
else:
    # 1. On crée une colonne 'annee' en prenant les 4 premiers caractères de 't'
    df_complet['annee'] = df_complet['t'].str[:4]

    # 2. On calcule la moyenne de l'indice par année et par commune
    df_annuel = df_complet.groupby(['l_codinsee', 'annee'])['Index_Final'].mean().reset_index()

    # 3. Création de la carte interactive magique
    # On s'assure que france_map est chargé (cellule GrgYVpKlzjtk)
    if 'france_map' in globals():
        fig = px.choropleth(df_annuel,
                            geojson=france_map,
                            locations='l_codinsee',
                            featureidkey="properties.code",
                            color='Index_Final',
                            animation_frame='annee',
                            color_continuous_scale="RdYlGn_r",
                            range_color=[90, df_annuel['Index_Final'].max()],
                            title="Évolution annuelle des prix immobiliers sur la Côte d'Azur (Base 100 en 2010)",
                            labels={'Index_Final': 'Indice de prix'})

        # 4. On zoome automatiquement sur la Côte d'Azur
        fig.update_geos(fitbounds="locations", visible=False)

        # 5. On affiche la carte dans Colab
        fig.show()

        # Sauvegarde en HTML
        fig.write_html("carte_interactive_cotedazur.html")
        print("✅ Carte générée et sauvegardée en HTML !")
    else:
        print("❌ ERREUR : 'france_map' est absent. Exécute la cellule de simplification des contours (GrgYVpKlzjtk).")

In [ ]:
from google.colab import drive

# 1. On connecte Google Drive à la session Colab
drive.mount('/content/drive')

print("Le coffre-fort est ouvert !")

In [ ]:
import os

# 2. On crée un dossier spécifique dans ton Google Drive
dossier_projet = '/content/drive/MyDrive/Projet_Immobilier_Master1'

# Si le dossier n'existe pas encore, Python le crée tout seul
if not os.path.exists(dossier_projet):
    os.makedirs(dossier_projet)
    print("Nouveau dossier créé dans ton Drive !")

# 3. On sauvegarde la base de données finale (celle avec les indices calculés)
# Comme ça, demain, tu pourras charger directement ce fichier !
fichier_csv_final = f"{dossier_projet}/base_immobiliere_finale.csv"
df_complet.to_csv(fichier_csv_final, index=False)
print("Base de données sauvegardée avec succès.")

# 4. On sauvegarde ta carte interactive HTML pour ton GitHub
fichier_carte = f"{dossier_projet}/carte_interactive_cotedazur.html"
fig.write_html(fichier_carte)
print("Carte HTML sauvegardée avec succès.")

# 5. On sauvegarde aussi la figure de la Heatmap (si tu l'avais appelée fig_heatmap par exemple)
# plt.savefig(f"{dossier_projet}/heatmap_communes.png", dpi=300)

print(f"Tout est en sécurité ! Va vérifier dans ton Google Drive, dans le dossier 'Projet_Immobilier_Master1'.")